<font size="6" color='grey'> <b>
KI und Generative AI. Verstehen. Anwenden. Gestalten.
</b></font> </br>

<font size="5" color='grey'> <b>
quick template
</b></font> </br>

---

In [ ]:
#@title 🔧 Umgebung einrichten{ display-mode: "form" }
!uv pip install --system -q git+https://github.com/ralf-42/GenAI.git#subdirectory=04_modul

from genai_lib.utilities import (
    check_environment,
    get_ipinfo,
    setup_api_keys,
    mprint,
    install_packages,
    mermaid,
    get_model_profile,
    extract_thinking,
    load_prompt,
)

setup_api_keys(['OPENAI_API_KEY'], create_globals=False)
print()
check_environment()
print()
get_ipinfo()

# 1 | Importe
---

In [ ]:
# ── Standardbibliotheken ──────────────────────────────────────────────
from typing import Optional

# ── LangChain ─────────────────────────────────────────────────────────────
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

# ── Strukturierte Ausgaben ──────────────────────────────────────────────────
from pydantic import BaseModel, Field

# ── Weitere Bibliotheken (nach Bedarf einkommentieren) ──────────────────────
# from langchain_chroma import Chroma
# from langchain_openai import OpenAIEmbeddings
# from langchain_core.documents import Document
from IPython.display import Image as IPImage, display

In [ ]:
# ── Modell ────────────────────────────────────────────────────────────────
# Modell-Auswahl: siehe GenAI/docs/frameworks/modell-auswahl-guide.md
# Demo/Grundlagen:   init_chat_model("openai:gpt-4o-mini", temperature=0.0)
# Worker/Content:    init_chat_model("openai:gpt-5.1")
# Structured Output: init_chat_model("openai:gpt-4o-mini", temperature=0.0)

llm = init_chat_model("openai:gpt-4o-mini", temperature=0.0)

# 2 | Chain — `prompt | llm | parser`
---

**Wann verwenden?** Für einzelne, zustandslose LLM-Aufrufe ohne Tools und ohne Gedächtnis.

| Einsatz | Beispiel |
|---------|---------|
| Texttransformation | Zusammenfassen, Übersetzen, Umschreiben |
| Klassifikation | Sentiment, Kategorie, Priorität |
| Extraktion | Entities, Keywords, Datum |

> Kein Gedächtnis · Keine Tools · Keine Verzweigung → `chain.invoke()`

In [ ]:
# ── ChatPromptTemplate ────────────────────────────────────────────────────
prompt = ChatPromptTemplate.from_messages([
    ("system", "Du bist ein hilfreicher Assistent. Antworte auf Deutsch."),
    ("human", "{input}"),
])

In [ ]:
# ── Parser ───────────────────────────────────────────────────────────────
parser = StrOutputParser()

In [ ]:
# ── Chain ────────────────────────────────────────────────────────────────
chain = prompt | llm | parser

In [ ]:
result = chain.invoke({"input": "Hallo!"})
mprint(result)

# 3 | Structured Output — `llm.with_structured_output()`
---

**Wann verwenden?** Wenn die Antwort des LLM eine definierte Datenstruktur haben soll.

| Einsatz | Beispiel |
|---------|---------|
| Extraktion | Entitäten, Metadaten, Fakten aus Text |
| Klassifikation mit Begründung | Kategorie + Confidence + Erklärung |
| Formularausfüllung | Adresse, Name, Datum aus Freitext |

> Das LLM füllt das **Pydantic-Modell** automatisch aus — kein manuelles Parsen.

In [ ]:
# ── Pydantic-Modell definieren ────────────────────────────────────────────
class Zusammenfassung(BaseModel):
    titel:      str       = Field(description="Kurzer Titel (max. 10 Wörter)")
    inhalt:     str       = Field(description="Zusammenfassung in 2-3 Sätzen")
    stichworte: list[str] = Field(description="3-5 Schlüsselwörter")

# ── Structured LLM ────────────────────────────────────────────────────────
structured_llm = llm.with_structured_output(Zusammenfassung)

In [ ]:
# ── Aufruf ────────────────────────────────────────────────────────────────────────────
text = "LangChain ist ein Framework für die Entwicklung von Anwendungen mit großen Sprachmodellen."

result = structured_llm.invoke(f"Fasse folgenden Text zusammen: {text}")

mprint(f"**Titel:** {result.titel}")
mprint(f"**Inhalt:** {result.inhalt}")
mprint(f"**Stichworte:** {', '.join(result.stichworte)}")

# 4 | Tool — `@tool`
---

**Wann verwenden?** Wenn der Agent externe Aktionen ausführen soll — Berechnungen, APIs, Datenbankzugriff.

| Einsatz | Beispiel |
|---------|---------|
| Berechnung | Mathematik, Währungsrechner, Statistik |
| Externe API | Wetter, Suche, Datenbank |
| Dateioperationen | Lesen, Schreiben, Verzeichnis |

> Der **Docstring** ist die Anweisung ans LLM — präzise formulieren, das LLM wählt das Tool danach aus.

In [ ]:
# ── Tool-Definition ─────────────────────────────────────────────────────
@tool
def mein_tool(eingabe: str) -> str:
    """[Beschreibung: Was macht dieses Tool? Wird direkt an das LLM übergeben.]"""
    # [Tool-Logik hier]
    return f"Ergebnis: {eingabe}"

# Tools testen
print(mein_tool.invoke({"eingabe": "Test"}))

# Tools-Liste
tools = [mein_tool]

# 5 | Agent — `create_react_agent`
---

**Wann verwenden?** Wenn das LLM selbst entscheiden soll, welche Tools es wann aufruft.

| Ansatz | Wann? |
|--------|-------|
| `chain` | Eine Anfrage, kein Zustand, keine Tools |
| `with_structured_output` | Strukturierte Daten extrahieren / klassifizieren |
| `create_react_agent` | LLM wählt Tools selbst — flexibel, mehrere Schritte |

> Der Agent plant eigenständig: Tool aufrufen → Ergebnis auswerten → nächsten Schritt wählen.

In [ ]:
from langgraph.prebuilt import create_react_agent

# ── Agent erstellen ────────────────────────────────────────────────────────────────────
agent = create_react_agent(
    model=llm,
    tools=tools,
    prompt="Du bist ein hilfreicher Assistent. Nutze die verfügbaren Tools. Antworte auf Deutsch.",
)

# ── Aufruf ────────────────────────────────────────────────────────────────────────────────
result = agent.invoke({"messages": [("human", "Teste das Tool mit 'Hallo Welt'.")]})
mprint(f"**Antwort:** {result['messages'][-1].content}")

# A | Aufgabe
---

Die Aufgabestellungen unten bieten Anregungen — eigene Herausforderungen sind ausdrücklich willkommen.

<p><font color='black' size="5">
[Aufgabentitel]
</font></p>

[Aufgabenbeschreibung]

**Teilaufgaben:**
- Teilaufgabe 1
- Teilaufgabe 2